# Literature screening, gene extraction, and validation workflow

This notebook documents the provenance chain used to construct the finalized
48-gene *Drosophila melanogaster* analytical dataset from the archived
507-record literature corpus.

## Workflow checkpoints

**507 bibliographic records → 18,679 extracted candidate mentions → 1,055 unique
gene-like candidates → 85 preliminarily validated genes → 48 finalized genes**

The literature-processing and candidate-extraction stages are computational.
The subsequent identifier/source validation and expert-curation stages are
audited against the archived validation datasets rather than reconstructed
with a retrospective selection rule.


## 1. Setup and input

The input corpus contains the bibliographic records retrieved in the literature search. The analysis uses `pandas` plus Python's standard `re` module.


In [ ]:
from pathlib import Path
import pandas as pd
import re

# Locate the repository root when run from the repository root or code/.
cwd = Path.cwd()

if (cwd / "data" / "literature_corpus_507_records.csv").exists():
    ROOT = cwd
elif (cwd.parent / "data" / "literature_corpus_507_records.csv").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Repository root could not be located. Expected "
        "data/literature_corpus_507_records.csv."
    )

INPUT_FILE = ROOT / "data" / "literature_corpus_507_records.csv"
RESULTS_DIR = ROOT / "results"
VALIDATION_85_FILE = ROOT / "data" / "Table_S1_gene_validation_85.csv"
FINAL_48_FILE = ROOT / "data" / "final_gene_set_48.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Archived provenance outputs.
DUPLICATE_AUDIT_FILE = RESULTS_DIR / "duplicate_groups_audit.csv"
UNIQUE_CANDIDATES_FILE = RESULTS_DIR / "gene_candidates_unique_1055.csv"

# Other intermediate outputs are kept separate from the principal archived results.
OUTPUT_DIR = RESULTS_DIR / "literature_screening_intermediate"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the frozen literature corpus.
df_raw = pd.read_csv(INPUT_FILE)

print(f"Repository root: {ROOT}")
print(f"Input: {INPUT_FILE}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Loaded records: {len(df_raw):,}")

if len(df_raw) != 507:
    raise ValueError(
        f"Expected 507 bibliographic records, but loaded {len(df_raw)}."
    )

df_raw.head()


## 2. Bibliographic deduplication

Records are clustered conservatively using exact normalized identifiers in the following hierarchy: PMID, DOI, PMCID, and an exact normalized informative title. Short generic headings such as `Posters`, `Poster Presentations`, or `Abstracts` are not used for title-based merging.

For duplicated records, database and search-query provenance and metal-search associations are retained as semicolon-separated values. One representative bibliographic record is retained for title/abstract text mining.

This step removes repeated database/search retrievals without treating a publication retrieved for more than one metal as multiple publications.


In [ ]:
def clean_text_id(value):
    if pd.isna(value):
        return None
    value = str(value).strip().lower()
    return value if value else None

def clean_pmid(value):
    if pd.isna(value):
        return None
    # PubMed exports in the historical corpus occasionally contain a MEDLINE
    # record after the PMID. The leading numeric PMID is extracted explicitly.
    match = re.match(r"\s*(\d{6,9})\b", str(value))
    return match.group(1) if match else None

def normalize_title(value):
    if pd.isna(value):
        return None
    text = str(value).lower()
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[^\w\s-]", "", text)
    return text or None

work = df_raw.copy()
work["_pmid_clean"] = work["pmid"].apply(clean_pmid)
work["_doi_clean"] = work["doi_norm"].apply(clean_text_id) if "doi_norm" in work else work["doi"].apply(clean_text_id)
work["_pmcid_clean"] = work["pmcid_norm"].apply(clean_text_id) if "pmcid_norm" in work else work["pmcid"].apply(clean_text_id)
work["_title_clean"] = work["title_norm"].apply(clean_text_id) if "title_norm" in work else work["title"].apply(normalize_title)

parent = list(range(len(work)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# Exact high-specificity identifiers.
for col in ["_pmid_clean", "_doi_clean", "_pmcid_clean"]:
    seen = {}
    for idx, value in work[col].items():
        if not value:
            continue
        if value in seen:
            union(idx, seen[value])
        else:
            seen[value] = idx

# Exact informative-title fallback.
generic_titles = {
    "posters", "poster presentations", "abstracts", "symposia",
    "oral presentations", "meeting abstracts"
}
seen_titles = {}
for idx, title in work["_title_clean"].items():
    if not title or len(title) < 25 or title in generic_titles:
        continue
    if title in seen_titles:
        union(idx, seen_titles[title])
    else:
        seen_titles[title] = idx

clusters = {}
for idx in range(len(work)):
    clusters.setdefault(find(idx), []).append(idx)

duplicate_groups = [indices for indices in clusters.values() if len(indices) > 1]

print(f"Unique publication-level records: {len(clusters):,}")
print(f"Redundant retrieval records removed: {len(work) - len(clusters):,}")
print(f"Duplicate groups: {len(duplicate_groups):,}")

assert len(clusters) == 454, "Unexpected deduplicated record count."


In [ ]:
def join_unique(series):
    values = []
    for value in series.dropna():
        value = str(value).strip()
        if value and value not in values:
            values.append(value)
    return "; ".join(values)

deduplicated_rows = []
audit_rows = []

for group_id, indices in enumerate(clusters.values(), start=1):
    subset = work.loc[indices]
    representative = subset.iloc[0].copy()

    representative["database"] = join_unique(subset["database"])
    representative["metal"] = join_unique(subset["metal"])
    representative["query"] = join_unique(subset["query"])
    representative["n_retrieval_records"] = len(indices)
    representative["dedup_group_id"] = group_id

    deduplicated_rows.append(representative)

    if len(indices) > 1:
        for idx in indices:
            audit_rows.append({
                "dedup_group_id": group_id,
                "source_row": int(idx) + 1,
                "database": work.at[idx, "database"],
                "metal": work.at[idx, "metal"],
                "query": work.at[idx, "query"],
                "pmid_clean": work.at[idx, "_pmid_clean"],
                "doi_clean": work.at[idx, "_doi_clean"],
                "pmcid_clean": work.at[idx, "_pmcid_clean"],
                "title": work.at[idx, "title"],
            })

df = pd.DataFrame(deduplicated_rows).reset_index(drop=True)
duplicate_audit = pd.DataFrame(audit_rows)

dedup_file = OUTPUT_DIR / "literature_corpus_deduplicated.csv"
audit_file = DUPLICATE_AUDIT_FILE

df.to_csv(dedup_file, index=False)
duplicate_audit.to_csv(audit_file, index=False)

print(f"Saved: {dedup_file}")
print(f"Saved: {audit_file}")


## 3. Semi-automatic literature screening

The original screening rules are retained. The first pass uses title, abstract, and the search-query field to classify records as `yes`, `maybe`, or `no`. Final priority classification uses title + abstract only, preventing search-query wording from determining high-priority status.


In [ ]:
df["screen_text"] = (
    df["title"].fillna("").astype(str) + " " +
    df["abstract"].fillna("").astype(str) + " " +
    df["query"].fillna("").astype(str)
).str.lower()

positive_drosophila = [
    "drosophila",
    "drosophila melanogaster"
]

positive_metals = [
    "cadmium",
    "lead",
    "mercury",
    "heavy metal",
    "metal stress"
]

strong_gene = [
    "gene expression",
    "transcriptome",
    "transcriptomic",
    "toxicogenomic",
    "rna-seq",
    "differential expression",
    "mutant",
    "knockout",
    "rnai",
    "metallothionein",
    "mtf-1",
    "ortholog"
]

moderate_gene = [
    "gene",
    "genes",
    "expression",
    "transcript",
    "pathway",
    "oxidative stress",
    "neuronal",
    "synaptic",
    "apoptosis",
    "mitochondria"
]

negative = [
    "mouse",
    "mice",
    "rat",
    "rats",
    "zebrafish",
    "arabidopsis",
    "plant"
]

def has_any(text, words):
    return any(w in text for w in words)

def classify_strict(text):
    dros = has_any(text, positive_drosophila)
    metal = has_any(text, positive_metals)
    strong = has_any(text, strong_gene)
    moderate = has_any(text, moderate_gene)
    neg = has_any(text, negative)

    if neg and not dros:
        return "no", "not drosophila / likely other model"
    if dros and metal and strong:
        return "yes", "drosophila + metal + strong gene-level evidence"
    if dros and metal and moderate:
        return "maybe", "drosophila + metal + possible molecular relevance"
    return "no", "insufficient relevance"

df[["include_auto", "reason_auto"]] = df["screen_text"].apply(
    lambda x: pd.Series(classify_strict(x))
)

order = {"yes": 0, "maybe": 1, "no": 2}
df["sort_key"] = df["include_auto"].map(order)

df_sorted = df.sort_values(["sort_key", "metal", "year"], ascending=[True, True, False])

df_sorted["include_final"] = ""
df_sorted["reason_final"] = ""
df_sorted["genes_found"] = ""

print(df_sorted["include_auto"].value_counts())
df_sorted[["database", "metal", "year", "title", "include_auto", "reason_auto"]].head(30)

high_priority_terms = [
    "gene expression", "transcriptome", "transcriptomic", "toxicogenomic",
    "rna-seq", "differential expression", "mutant", "knockout", "rnai",
    "metallothionein", "mtf-1", "ortholog", "pathway", "gene", "genes"
]

review_terms = [
    "review", "lessons from drosophila", "model for", "mechanisms"
]

low_priority_terms = [
    "lifespan", "survival", "reproductive fitness", "locomotion",
    "motor ability", "behavior", "behaviour"
]

def priority_class(text):
    txt = str(text).lower()
    high = any(t in txt for t in high_priority_terms)
    review = any(t in txt for t in review_terms)
    low = any(t in txt for t in low_priority_terms)

    if high:
        return "high_priority"
    if review:
        return "medium_priority"
    if low:
        return "low_priority"
    return "medium_priority"

# применяем только к тем, что уже yes или maybe
df_sorted["priority_auto"] = ""

mask = df_sorted["include_auto"].isin(["yes", "maybe"])
df_sorted.loc[mask, "priority_auto"] = df_sorted.loc[mask, "screen_text"].apply(priority_class)

print(df_sorted.loc[mask, "priority_auto"].value_counts())

df_sorted.loc[mask, ["database", "metal", "year", "title", "include_auto", "priority_auto"]].head(50)

# Final priority classification uses title + abstract only.
# только title + abstract, без query
df_sorted["screen_text_clean"] = (
    df_sorted["title"].fillna("").astype(str) + " " +
    df_sorted["abstract"].fillna("").astype(str)
).str.lower()

strong_terms = [
    "gene expression",
    "transcriptome",
    "transcriptomic",
    "toxicogenomic",
    "rna-seq",
    "differential expression",
    "mutant",
    "knockout",
    "rnai",
    "metallothionein",
    "mtf-1",
    "ortholog"
]

moderate_terms = [
    "expression",
    "transcript",
    "pathway",
    "oxidative stress",
    "neuronal",
    "synaptic",
    "apoptosis",
    "mitochondria"
]

review_terms = [
    "review",
    "lessons from drosophila",
    "model for",
    "mechanisms"
]

low_priority_terms = [
    "lifespan",
    "survival",
    "reproductive fitness",
    "locomotion",
    "motor ability",
    "behavior",
    "behaviour"
]

def has_any(text, words):
    return any(w in text for w in words)

def priority_class(text):
    txt = str(text).lower()
    strong = has_any(txt, strong_terms)
    moderate = has_any(txt, moderate_terms)
    review = has_any(txt, review_terms)
    low = has_any(txt, low_priority_terms)

    if strong:
        return "high_priority"
    if moderate or review:
        return "medium_priority"
    if low:
        return "low_priority"
    return "low_priority"

mask = df_sorted["include_auto"].isin(["yes", "maybe"])
df_sorted.loc[mask, "priority_auto"] = df_sorted.loc[mask, "screen_text_clean"].apply(priority_class)

print(df_sorted.loc[mask, "priority_auto"].value_counts())

df_sorted.loc[mask, ["database", "metal", "year", "title", "include_auto", "priority_auto"]].head(50)

print("Automated inclusion classes:")
print(df_sorted["include_auto"].value_counts())
print("\nPriority classes among yes/maybe records:")
print(df_sorted.loc[mask, "priority_auto"].value_counts())


print("\nFinal priority distribution:")
print(df_sorted.loc[mask, "priority_auto"].value_counts())


In [ ]:
screening_file = OUTPUT_DIR / "screening_semiauto_strict.csv"
df_sorted.to_csv(screening_file, index=False)
print(f"Saved: {screening_file}")


## 4. High-priority corpus and text preparation

Only records classified as `high_priority` enter gene-like token extraction. Candidate extraction is intentionally permissive; biological identity and metal association are established only during downstream validation.


In [ ]:
core_df = df_sorted[df_sorted["priority_auto"] == "high_priority"].copy()

core_df.shape

core_df["text_for_gene_search"] = (
    core_df["title"].fillna("").astype(str) + " " +
    core_df["abstract"].fillna("").astype(str)
)

print(f"High-priority records: {len(core_df):,}")


assert len(core_df) == 144, "Unexpected high-priority record count."


## 5. Gene-like token extraction

The original seed-symbol list and regular expressions are retained. These patterns identify gene-like textual tokens; they do not validate gene identity.


In [ ]:
import re
import pandas as pd

seed_genes = [
    "MTF-1", "MtnA", "MtnB", "MtnC", "MtnD",
    "Sod1", "Sod2", "Cat", "GstD1", "GstE1",
    "Hsp70", "Hsp83", "p53", "reaper", "hid",
    "para", "cac", "Sh", "Syn",
    "CncC", "Keap1", "JNK", "Thor"
]

def find_seed_genes(text, gene_list):
    text = str(text)
    found = []
    for gene in gene_list:
        pattern = r"\b" + re.escape(gene) + r"\b"
        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(gene)
    return sorted(set(found))

core_df["seed_genes_found"] = core_df["text_for_gene_search"].apply(
    lambda x: find_seed_genes(x, seed_genes)
)

core_df[["title", "seed_genes_found"]].head(10)

# простые стоп-слова, чтобы убрать очевидный шум
stop_words = {
    "DNA", "RNA", "ATP", "ROS", "Cd", "Pb", "Hg",
    "Drosophila", "Genes", "Gene", "Metal", "Stress",
    "Lead", "Mercury", "Cadmium", "Review"
}

def extract_gene_like_tokens(text):
    text = str(text)

    # шаблоны под короткие gene-like названия:
    patterns = [
        r"\b[A-Z][a-z]{1,4}\d{0,2}\b",      # Sod1, Cat, Syn
        r"\b[A-Z]{2,5}-\d\b",               # MTF-1
        r"\b[a-z]{2,10}\b"                  # para, reaper, hid, notch-like lowercase genes
    ]

    hits = []
    for pat in patterns:
        hits.extend(re.findall(pat, text))

    # чистка
    cleaned = []
    for h in hits:
        if h in stop_words:
            continue
        if len(h) < 3:
            continue
        cleaned.append(h)

    return sorted(set(cleaned))

core_df["gene_like_tokens"] = core_df["text_for_gene_search"].apply(extract_gene_like_tokens)

core_df[["title", "gene_like_tokens"]].head(10)

def merge_gene_lists(row):
    merged = set(row["seed_genes_found"]) | set(row["gene_like_tokens"])
    return sorted(merged)

core_df["candidate_genes_auto"] = core_df.apply(merge_gene_lists, axis=1)


## 6. Publication-level candidate mentions

Candidate tokens are expanded to long format and deduplicated within each publication-level metal-association/title/token combination.


In [ ]:
rows = []

for _, row in core_df.iterrows():
    for gene in row["candidate_genes_auto"]:
        rows.append({
            "metal": row.get("metal", ""),
            "database_found": row.get("database", ""),
            "search_query": row.get("query", ""),
            "title": row.get("title", ""),
            "year": row.get("year", ""),
            "doi": row.get("doi", ""),
            "pmid": row.get("pmid", ""),
            "gene_symbol_raw": gene,
            "evidence_stage": "auto_extracted_candidate",
            "evidence_type": "title_abstract_text_mining",
            "validation_status": "",
            "gene_symbol_validated": "",
            "flybase_id": "",
            "known_function": "",
            "putative_role_in_neurotoxicity": "",
            "include_final": ""
        })

gene_candidates_df = pd.DataFrame(rows)
gene_candidates_df.head()
gene_candidates_df.shape

gene_candidates_df["gene_symbol_raw_norm"] = (
    gene_candidates_df["gene_symbol_raw"]
    .astype(str)
    .str.strip()
    .str.lower()
)

gene_candidates_df = gene_candidates_df.drop_duplicates(
    subset=["metal", "title", "gene_symbol_raw_norm"]
).copy()

gene_candidates_df.shape

print(f"Candidate mentions after record-level deduplication: {len(gene_candidates_df):,}")
assert len(gene_candidates_df) == 18679, "Expected 18,679 candidate mentions after bibliographic deduplication."


In [ ]:
candidate_file = OUTPUT_DIR / "gene_candidates_auto.csv"
gene_candidates_df.to_csv(candidate_file, index=False)
print(f"Saved: {candidate_file}")


## 7. Unique gene-like candidate set

Syntax and stop-word filters from the original workflow are applied. The former automated relevance-shortlisting stage is not used because it depended on which context happened to be encountered first for a token. Instead, all unique gene-like candidates passing the deterministic token filters are forwarded to identifier/evidence validation.


In [ ]:
df = gene_candidates_df.copy()

# 1. длина >= 3
df = df[df["gene_symbol_raw"].str.len() >= 3]

# 2. убираем слова с маленькой буквы (оставляем gene-like)
df = df[df["gene_symbol_raw"].str.match(r'^[A-Za-z0-9\-]+$')]

# 3. убираем слова с пробелами
df = df[~df["gene_symbol_raw"].str.contains(" ")]

# 4. убираем полностью lowercase слова (часто шум)
df = df[~df["gene_symbol_raw"].str.islower()]

df.shape

stop_words = [
    "Biol", "Dev", "New", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    "Article", "Study", "Effect", "Role", "Analysis",
    "Data", "Results", "Using", "Based"
]

df = df[~df["gene_symbol_raw"].isin(stop_words)]

df.shape

df["gene_symbol_raw_norm"] = df["gene_symbol_raw"].str.lower()

df_unique = df.drop_duplicates(subset=["gene_symbol_raw_norm"]).copy()

df_unique.shape

print(f"Unique tokens after first-stage filtering: {len(df_unique):,}")


In [ ]:
df = df_unique.copy()

df = df[df["gene_symbol_raw"].notna()].copy()
df = df[df["gene_symbol_raw"].astype(str).str.lower() != "none"].copy()
df["gene_symbol_raw"] = df["gene_symbol_raw"].astype(str).str.strip()
df["gene_symbol_norm"] = df["gene_symbol_raw"].str.lower()

stop_words = {
    "biol", "dev", "new", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "article", "study", "effect", "effects", "role", "analysis", "data", "results", "using",
    "stress", "metal", "metals", "heavy", "cadmium", "lead", "mercury", "drosophila",
    "gene", "genes", "pathway", "neurotoxicity", "cell", "cells", "protein", "proteins",
    "dna", "rna", "atp", "ros"
}

df = df[df["gene_symbol_raw"].str.len() >= 3].copy()
df = df[df["gene_symbol_raw"].str.match(r"^[A-Za-z0-9\-]+$", na=False)].copy()
df = df[~df["gene_symbol_norm"].isin(stop_words)].copy()
df = df[~df["gene_symbol_raw"].str.match(r"^\d+$", na=False)].copy()

def gene_like_rule(symbol):
    s = str(symbol)
    patterns = [
        r"^[A-Z][a-z]{1,4}\d{0,2}$",
        r"^[A-Z]{2,5}-\d$",
        r"^[A-Z][A-Za-z]{1,8}$",
        r"^[a-z]{3,10}$",
        r"^[A-Za-z]{2,8}\d{1,3}$"
    ]
    return any(re.match(pattern, s) for pattern in patterns)

df["is_gene_like"] = df["gene_symbol_raw"].apply(gene_like_rule)
gene_candidates_unique = (
    df[df["is_gene_like"]]
    .drop_duplicates(subset=["gene_symbol_norm"])
    .sort_values("gene_symbol_norm")
    .reset_index(drop=True)
)

print(f"Unique gene-like candidates: {len(gene_candidates_unique):,}")
assert len(gene_candidates_unique) == 1055, "Expected 1,055 unique gene-like candidates."


In [ ]:
unique_file = UNIQUE_CANDIDATES_FILE
gene_candidates_unique.to_csv(unique_file, index=False)
print(f"Saved: {unique_file}")


## 8. Reproducibility summary

Expected checkpoints for the revised workflow:

- retrieved bibliographic records: **507**;
- unique publication-level records after explicit deduplication: **454**;
- high-priority publication-level records: **144**;
- gene-like textual mentions after publication-level processing: **18,679**;
- unique gene-like candidates forwarded to identifier/evidence validation: **1,055**.

The subsequent transition from 1,055 gene-like candidates to 85 identifier-validated candidates involved identifier/source validation and is documented separately. The evidence-based reassessment of those 85 candidates to the final 48-gene analytical set is provided in `data/Table_S1_gene_validation_85.csv`.

This notebook intentionally does not infer pathway activation, causal toxicological effects, or gene validity from text-mining frequency.


## Archived validation stages: 1,055 → 85 → 48

The transition from the automated gene-like candidate pool to the preliminary
85-gene set involved identifier/source validation and expert curation. This
notebook does not invent a retrospective algorithm to force that manual stage
to reproduce a predetermined count.

Instead, it loads the archived 85-gene validation table and the frozen
48-gene dataset and verifies their cardinalities and identifier consistency.
The relevant files are:

- `data/Table_S1_gene_validation_85.csv`
- `data/final_gene_set_48.csv`


In [ ]:
# Audit archived validation stages.
validation_85 = pd.read_csv(
    VALIDATION_85_FILE,
    dtype=str,
    keep_default_na=False
)

final_48 = pd.read_csv(
    FINAL_48_FILE,
    dtype=str,
    keep_default_na=False
)

print(f"Preliminary validation table: {len(validation_85):,} rows")
print(f"Final analytical dataset: {len(final_48):,} rows")

assert len(validation_85) == 85, (
    f"Expected 85 rows in validation table; found {len(validation_85)}."
)
assert len(final_48) == 48, (
    f"Expected 48 rows in final dataset; found {len(final_48)}."
)

# Use the explicit identifier columns defined in the archived datasets.
ids85 = set(
    validation_85["FlyBase ID"]
    .str.strip()
    .loc[lambda x: x != ""]
)
ids48 = set(
    final_48["FlyBase_ID"]
    .str.strip()
    .loc[lambda x: x != ""]
)

print(f"Unique FlyBase IDs in 85-gene table: {len(ids85)}")
print(f"Unique FlyBase IDs in final set: {len(ids48)}")

assert len(ids85) == 85, (
    f"Expected 85 unique FlyBase IDs; found {len(ids85)}."
)
assert len(ids48) == 48, (
    f"Expected 48 unique FlyBase IDs; found {len(ids48)}."
)

missing = sorted(ids48 - ids85)
print(f"Final-set IDs absent from 85-gene table: {len(missing)}")

if missing:
    raise ValueError(
        f"Final-set IDs absent from validation table: {missing}"
    )

print("Archived validation-stage audit passed: 85 → 48.")


## Provenance summary

The notebook distinguishes computational reproduction of the literature and
candidate-extraction stages from audit reproduction of the subsequent
identifier/source-validation and expert-curation stages. No retrospective
algorithm is introduced to manufacture the archived 85- and 48-gene counts.
